# EpiScope Tutorial (Reconstructed): Explorer Mode

This reconstructed notebook replaces the original (which was corrupted with ellipses and invalid JSON). It demonstrates a **self-contained Explorer**-style workflow:

- Load a small in-memory corpus of documents
- Build lightweight bag-of-words vectors (no external dependencies)
- Retrieve the top-k relevant passages for a query
- Compose a tiny, rule-based "answer" grounded in the retrieved context

**Why the original failed:** the uploaded `.ipynb` contained literal `...` ellipses and missing characters inside JSON strings, making it invalid JSON and impossible to open.

> This version is designed to run anywhere with standard Python + `numpy` available (no servers, no models, no vector DB).

In [1]:
# Minimal dependencies
import re, math
from collections import Counter
from typing import List, Tuple
import numpy as np

# Tiny toy corpus (title, text)
CORPUS = [
    ("Influenza Overview", "Influenza is a contagious respiratory illness caused by influenza viruses. Symptoms include fever, cough, and sore throat."),
    ("Vaccination Guidance", "Annual flu vaccination is recommended. Vaccines reduce the risk of severe disease and hospitalization."),
    ("Non-Pharmaceutical Interventions", "Hand hygiene, masking, and isolation when ill can reduce transmission of respiratory viruses."),
    ("Antiviral Treatments", "Antivirals like oseltamivir may shorten illness if started early, especially for high-risk groups."),
]

# Simple preprocessing
_token_re = re.compile(r"[a-zA-Z0-9']+")

def tokenize(text: str):
    return [t.lower() for t in _token_re.findall(text)]

# Build vocabulary
all_tokens = []
for _, txt in CORPUS:
    all_tokens.extend(tokenize(txt))
vocab = sorted(set(all_tokens))
index = {t:i for i,t in enumerate(vocab)}

# Vectorization (bag-of-words)
def vec(text: str):
    v = np.zeros(len(vocab), dtype=float)
    for tok, c in Counter(tokenize(text)).items():
        i = index.get(tok)
        if i is not None:
            v[i] = c
    return v

DOC_VECS = [vec(txt) for _, txt in CORPUS]

# Cosine similarity
def cosim(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# Retrieve top-k
def retrieve(query: str, k: int = 2) -> List[Tuple[str, str, float]]:
    qv = vec(query)
    scored = []
    for (title, txt), dv in zip(CORPUS, DOC_VECS):
        scored.append((title, txt, cosim(qv, dv)))
    return sorted(scored, key=lambda x: x[2], reverse=True)[:k]

# Grounded mini-answer: stitches sentences that contain the top tokens
def make_answer(query: str, hits: List[Tuple[str, str, float]]) -> str:
    key = set(tokenize(query))
    sentences = []
    for _, txt, _ in hits:
        for s in re.split(r"(?<=[.!?])\s+", txt.strip()):
            if any(w in tokenize(s) for w in key):
                sentences.append(s)
    if not sentences:
        # fallback: first sentence of best hit
        sentences = [re.split(r"(?<=[.!?])\s+", hits[0][1].strip())[0]] if hits else ["No evidence found in corpus."]
    return " ".join(dict.fromkeys(sentences))  # dedupe while preserving order

print("Ready. Try setting `QUERY` and running retrieval.")

Ready. Try setting `QUERY` and running retrieval.


In [2]:
# Example query
QUERY = "How to reduce transmission of respiratory viruses?"
results = retrieve(QUERY, k=2)
for i, (title, txt, score) in enumerate(results, start=1):
    print(f"Hit {i}: {title} (score={score:.3f})\n  {txt}\n")
print("Suggested answer:\n", make_answer(QUERY, results))

Hit 1: Non-Pharmaceutical Interventions (score=0.620)
  Hand hygiene, masking, and isolation when ill can reduce transmission of respiratory viruses.

Hit 2: Vaccination Guidance (score=0.239)
  Annual flu vaccination is recommended. Vaccines reduce the risk of severe disease and hospitalization.

Suggested answer:
 Hand hygiene, masking, and isolation when ill can reduce transmission of respiratory viruses. Vaccines reduce the risk of severe disease and hospitalization.
